# Street weather features

This notebook assigns each street to a nearest weather station and builds the hourly and daily street weather tables. Outputs: `data/derived/street_weather.csv` and `data/derived/street_weather_daily.csv`.


In [1]:
import pandas as pd
import numpy as np

cscl = pd.read_csv('data/reference/CSCL.csv')

stations = np.array([
    [40.703, -74.006],  # 1 City Hall
    [40.597, -73.916],  # 2 Brooklyn Salt Marsh
    [40.649, -73.790],  # 3 JFK
    [40.774, -73.879],  # 4 LGA
    [40.849, -73.878],  # 5 Bronx Zoo
    [40.588, -74.140],  # 6 Staten Island
])

def first_lon_lat_from_wkt(the_geom_wkt):
    if not isinstance(the_geom_wkt, str):
        return None

    text = the_geom_wkt.strip()
    if not text or 'EMPTY' in text.upper() or '(' not in text:
        return None

    if '((' in text:
        coord_text = text.split('((', 1)[1]
    else:
        coord_text = text.split('(', 1)[1]

    coord_text = coord_text.split(',', 1)[0].split(')', 1)[0].strip()
    parts = coord_text.split()
    if len(parts) < 2:
        return None

    try:
        lon = float(parts[0])
        lat = float(parts[1])
    except ValueError:
        return None

    return lon, lat

def nearest_station(the_geom_wkt):
    coord = first_lon_lat_from_wkt(the_geom_wkt)
    if coord is None:
        return None

    lon, lat = coord
    dists = np.sqrt((stations[:, 0] - lat) ** 2 + (stations[:, 1] - lon) ** 2)
    return int(np.argmin(dists) + 1)

cscl['nearest_station'] = cscl['the_geom'].apply(nearest_station)


In [2]:
from street_normalize import normalize

In [3]:
cscl["normalized_street_name"] = (
    cscl["Borough Code"].astype(str)
    + "-"
    + cscl["Full Street Name"].apply(normalize)
)

In [4]:
import numpy as np
import pandas as pd

# --- 1. Compute median values grouped by type (either POST_TYPE or PRE_TYPE) ---
# Flatten POST_TYPE and PRE_TYPE into a single "type" for median calculation
cscl_types = cscl.copy()
cscl_types["TYPE"] = cscl_types.apply(
    lambda r: r["POST_TYPE"] if pd.notna(r["POST_TYPE"]) else r["PRE_TYPE"], axis=1
)

medians = (
    cscl_types.groupby("TYPE")[["Segment Length", "Street Width"]]
    .median(numeric_only=True)
)

# --- 2. Function to impute by type ---
def impute_by_type(row):
    # Determine the type to use
    types_present = []
    if pd.notna(row["POST_TYPE"]):
        types_present.append(row["POST_TYPE"])
    if pd.notna(row["PRE_TYPE"]):
        types_present.append(row["PRE_TYPE"])
    
    if len(types_present) == 1:
        t = types_present[0]
    elif len(types_present) == 2:
        if types_present[0] == types_present[1]:
            t = types_present[0]
        else:
            t = None  # conflict, cannot impute
    else:
        t = None  # neither populated
    
    if t and t in medians.index:
        if pd.isna(row["Segment Length"]):
            row["Segment Length"] = medians.loc[t, "Segment Length"]
        if pd.isna(row["Street Width"]):
            row["Street Width"] = medians.loc[t, "Street Width"]
    return row

# Apply imputation
cscl = cscl.apply(impute_by_type, axis=1)

# --- 3. Handle rows where type info missing but Segment Length or Street Width is missing ---
mask_missing_type = (cscl["POST_TYPE"].isna() & cscl["PRE_TYPE"].isna()) & (
    cscl["Segment Length"].isna() | cscl["Street Width"].isna()
)
problem_rows = cscl[mask_missing_type].copy()

In [5]:
cscl = cscl[["PHYSICALID", "normalized_street_name", "Segment Length", "Street Width", "nearest_station"]]

In [6]:
# dictionary normalized street name: all physical id's with that street name

name_to_ids = (
    cscl.groupby("normalized_street_name")["PHYSICALID"]
    .apply(list)
    .to_dict()
)

new df called street_weather constructed as follows: it has a "normalized_street_name" column, and columns "DATA 20XX" where DATA is in {temperature, apparent_temperature, snowfall, snow_depth, precipitation, relative_humidity, cloud_cover, wind_gusts_10m, wind_speed_10m}. normalized_street_name goes over all those keys from the dictionary name_to_ids. all other values are computed as the average value across (*) entries in the column whose name begins with that given string (e.g. temperature will be the start of actual string name "temperate_2m (degrees F)" in the corresponding csv) in weather_hourly_i where i is the value in nearest_station in cscl df for that normalized_street_name (might not be unique row, just choose any row with that normalized_street_name), where (*) means cover november 21 20XX-1 to march 20 20XX (e.g. november 21 2015 to march 20 2016 for 20XX = 2016), and 20XX goes from 2016 to 2025

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

# --- Config ---
weather_dir = Path("data/raw/weather/hourly")
YEARS = range(2016, 2026)
VARIABLES = [
    "temperature",
    "apparent_temperature",
    "snowfall",
    "snow_depth",
    "precipitation",
    "relative_humidity",
    "cloud_cover",
    "wind_gusts_10m",
    "wind_speed_10m",
]

# --- Load weather_hourly_i CSVs ---
weather_hourly = {}
for i in range(1, 7):
    f = weather_dir / f"weather_hourly_{i}.csv"
    df = pd.read_csv(f, skiprows=3)
    # Try to find datetime column heuristically
    datetime_col = next(c for c in df.columns if "time" in c.lower())
    df[datetime_col] = pd.to_datetime(df[datetime_col], errors="coerce")
    df.rename(columns={datetime_col: "datetime"}, inplace=True)
    weather_hourly[i] = df

# --- Helper to match a variable prefix ---
def get_column(df, prefix):
    prefix = prefix.lower()
    for c in df.columns:
        if c.lower().startswith(prefix):
            return c
    raise KeyError(f"No column starts with '{prefix}' in {df.columns[:8]}...")

# --- Map streets to nearest station ---
station_map = cscl.groupby("normalized_street_name")["nearest_station"].first().to_dict()

records = []

for street, ids in name_to_ids.items():
    if street not in station_map:
        continue

    station_id = station_map[street]
    if station_id not in weather_hourly:
        continue

    wdf = weather_hourly[station_id]

    for year in YEARS:
        start = pd.Timestamp(year - 1, 11, 21)
        end = pd.Timestamp(year, 3, 20, 23, 59)
        sub = wdf.loc[(wdf["datetime"] >= start) & (wdf["datetime"] <= end)]

        if sub.empty:
            continue

        entry = {"normalized_street_name": street}
        for var in VARIABLES:
            col = get_column(wdf, var)
            entry[f"{var} {year}"] = sub[col].mean(skipna=True)
        records.append(entry)

# --- Construct final DataFrame ---
street_weather = pd.DataFrame(records)
street_weather = street_weather.groupby("normalized_street_name").first().reset_index()


In [8]:
# Melt wide columns (e.g. temperature 2016, snowfall 2016, …) into long form

street_weather_melted = street_weather.melt(
    id_vars=['normalized_street_name'],
    var_name='variable_season',
    value_name='value'
)

# Split variable and season
street_weather_melted[['variable', 'season']] = street_weather_melted['variable_season'].str.extract(r'(.+)\s(\d{4})')

# Create combined identifier
street_weather_melted['normalized_street_name_season'] = (
    street_weather_melted['normalized_street_name'] + '_' + street_weather_melted['season']
)

# Pivot back so that each variable becomes a column
street_weather_long = street_weather_melted.pivot_table(
    index='normalized_street_name_season',
    columns='variable',
    values='value',
    aggfunc='first'
).reset_index()

# Optional: flatten columns
street_weather_long.columns.name = None

# Result:
# columns → ['normalized_street_name_season', 'temperature', 'apparent_temperature', 'snowfall', ...]


In [9]:
street_weather_long.to_csv("data/derived/street_weather.csv", index=False)

This will create `data/derived/street_weather_daily.csv`.

In [10]:
import pandas as pd
import numpy as np

cscl = pd.read_csv('data/reference/CSCL.csv')

stations = np.array([
    [40.703, -74.006],  # 1 City Hall
    [40.597, -73.916],  # 2 Brooklyn Salt Marsh
    [40.649, -73.790],  # 3 JFK
    [40.774, -73.879],  # 4 LGA
    [40.849, -73.878],  # 5 Bronx Zoo
    [40.588, -74.140],  # 6 Staten Island
])

def first_lon_lat_from_wkt(the_geom_wkt):
    if not isinstance(the_geom_wkt, str):
        return None

    text = the_geom_wkt.strip()
    if not text or 'EMPTY' in text.upper() or '(' not in text:
        return None

    if '((' in text:
        coord_text = text.split('((', 1)[1]
    else:
        coord_text = text.split('(', 1)[1]

    coord_text = coord_text.split(',', 1)[0].split(')', 1)[0].strip()
    parts = coord_text.split()
    if len(parts) < 2:
        return None

    try:
        lon = float(parts[0])
        lat = float(parts[1])
    except ValueError:
        return None

    return lon, lat

def nearest_station(the_geom_wkt):
    coord = first_lon_lat_from_wkt(the_geom_wkt)
    if coord is None:
        return None

    lon, lat = coord
    dists = np.sqrt((stations[:, 0] - lat) ** 2 + (stations[:, 1] - lon) ** 2)
    return int(np.argmin(dists) + 1)

cscl['nearest_station'] = cscl['the_geom'].apply(nearest_station)


In [11]:
from street_normalize import normalize

In [12]:
cscl["normalized_street_name"] = (
    cscl["Borough Code"].astype(str)
    + "-"
    + cscl["Full Street Name"].apply(normalize)
)

In [13]:
import numpy as np
import pandas as pd

# --- 1. Compute median values grouped by type (either POST_TYPE or PRE_TYPE) ---
# Flatten POST_TYPE and PRE_TYPE into a single "type" for median calculation
cscl_types = cscl.copy()
cscl_types["TYPE"] = cscl_types.apply(
    lambda r: r["POST_TYPE"] if pd.notna(r["POST_TYPE"]) else r["PRE_TYPE"], axis=1
)

medians = (
    cscl_types.groupby("TYPE")[["Segment Length", "Street Width"]]
    .median(numeric_only=True)
)

# --- 2. Function to impute by type ---
def impute_by_type(row):
    # Determine the type to use
    types_present = []
    if pd.notna(row["POST_TYPE"]):
        types_present.append(row["POST_TYPE"])
    if pd.notna(row["PRE_TYPE"]):
        types_present.append(row["PRE_TYPE"])
    
    if len(types_present) == 1:
        t = types_present[0]
    elif len(types_present) == 2:
        if types_present[0] == types_present[1]:
            t = types_present[0]
        else:
            t = None  # conflict, cannot impute
    else:
        t = None  # neither populated
    
    if t and t in medians.index:
        if pd.isna(row["Segment Length"]):
            row["Segment Length"] = medians.loc[t, "Segment Length"]
        if pd.isna(row["Street Width"]):
            row["Street Width"] = medians.loc[t, "Street Width"]
    return row

# Apply imputation
cscl = cscl.apply(impute_by_type, axis=1)

# --- 3. Handle rows where type info missing but Segment Length or Street Width is missing ---
mask_missing_type = (cscl["POST_TYPE"].isna() & cscl["PRE_TYPE"].isna()) & (
    cscl["Segment Length"].isna() | cscl["Street Width"].isna()
)
problem_rows = cscl[mask_missing_type].copy()

In [14]:
# dictionary normalized street name: all physical id's with that street name

name_to_ids = (
    cscl.groupby("normalized_street_name")["PHYSICALID"]
    .apply(list)
    .to_dict()
)

new df called street_weather constructed as follows: it has a "normalized_street_name" column, and columns "DATA SEASON 20XX" where DATA is in {temperature, rain, precipitation, snowfall, precipitation_hours, wind_speed}, and SEASON is in {winter, roadwork}. normalized_street_name goes over all those keys from the dictionary name_to_ids. all other values are computed as the average value across (*) entries in the column whose name begins with that given string (e.g. temperature will be the start of actual string name "temperate_2m (degrees F)" in the corresponding csv) in weather_hourly_i where i is the value in nearest_station in cscl df for that normalized_street_name (might not be unique row, just choose any row with that normalized_street_name), where (*) means cover november 21 20XX-1 to march 20 20XX (e.g. november 21 2015 to march 20 2016 for 20XX = 2016) for SEASON = winter, and march 21 20XX - november 20 20XX for SEASON = roadwork. and 20XX goes from 2001 to 2025.

In [15]:
import pandas as pd
import numpy as np
from pathlib import Path

# --- Config ---
weather_dir = Path("data/raw/weather/daily")
YEARS = range(2001, 2026)
VARIABLES = ["temperature", "rain", "precipitation", "snowfall", "precipitation_hours", "wind_speed"]
SEASONS = ["winter", "roadwork"]

# --- Load weather_hourly_i CSVs ---
weather_hourly = {}
for i in range(1, 7):  # adjust range to actual station IDs available
    f = weather_dir / f"weather_daily_{i}.csv"
    df = pd.read_csv(f, skiprows=3)
    datetime_col = next(c for c in df.columns if "time" in c.lower())
    df[datetime_col] = pd.to_datetime(df[datetime_col], errors="coerce")
    df.rename(columns={datetime_col: "datetime"}, inplace=True)
    weather_hourly[i] = df

# --- Helper to find first matching column prefix ---
def get_column(df, prefix):
    prefix = prefix.lower()
    for c in df.columns:
        if c.lower().startswith(prefix):
            return c
    raise KeyError(f"No column starts with '{prefix}' in {df.columns[:8]}...")

# --- Map streets to their nearest station ---
station_map = cscl.groupby("normalized_street_name")["nearest_station"].first().to_dict()

records = []

for street, ids in name_to_ids.items():
    if street not in station_map:
        continue

    station_id = station_map[street]
    if station_id not in weather_hourly:
        continue

    wdf = weather_hourly[station_id]

    for year in YEARS:
        # --- Winter season: Nov 21 (year-1) – Mar 20 (year) ---
        start_winter = pd.Timestamp(year - 1, 11, 21)
        end_winter = pd.Timestamp(year, 3, 20, 23, 59)
        sub_winter = wdf.loc[(wdf["datetime"] >= start_winter) & (wdf["datetime"] <= end_winter)]

        # --- Roadwork season: Mar 21 (year) – Nov 20 (year) ---
        start_roadwork = pd.Timestamp(year, 3, 21)
        end_roadwork = pd.Timestamp(year, 11, 20, 23, 59)
        sub_roadwork = wdf.loc[(wdf["datetime"] >= start_roadwork) & (wdf["datetime"] <= end_roadwork)]

        entry = {"normalized_street_name": street}

        for var in VARIABLES:
            try:
                col = get_column(wdf, var)
            except KeyError:
                continue

            if not sub_winter.empty:
                entry[f"{var} winter {year}"] = sub_winter[col].mean(skipna=True)
            if not sub_roadwork.empty:
                entry[f"{var} roadwork {year}"] = sub_roadwork[col].mean(skipna=True)

        records.append(entry)

# --- Construct final DataFrame ---
street_weather = pd.DataFrame(records)
street_weather = street_weather.groupby("normalized_street_name").first().reset_index()


In [16]:
street_weather.head()

,normalized_street_name,temperature winter 2001,temperature roadwork 2001,rain winter 2001,rain roadwork 2001,precipitation winter 2001,precipitation roadwork 2001,snowfall winter 2001,snowfall roadwork 2001,precipitation_hours winter 2001,...,rain winter 2025,rain roadwork 2025,precipitation winter 2025,precipitation roadwork 2025,snowfall winter 2025,snowfall roadwork 2025,precipitation_hours winter 2025,precipitation_hours roadwork 2025,wind_speed winter 2025,wind_speed roadwork 2025
0,1-1 avenue,30.739241,62.294694,1.348101,2.252245,2.383544,2.270612,0.740759,0.013143,3.493671,...,2.647500,3.078995,3.081667,3.089954,0.303917,0.007671,3.375000,3.757991,10.795833,9.381279
1,1-1 avenue loop,31.291139,62.564490,1.348101,2.252245,2.383544,2.270612,0.740759,0.013143,3.493671,...,2.759167,3.366210,3.157500,3.374886,0.278833,0.006073,3.341667,3.735160,10.518333,9.402283
2,1-1 drive,30.739241,62.294694,1.348101,2.252245,2.383544,2.270612,0.740759,0.013143,3.493671,...,2.647500,3.078995,3.081667,3.089954,0.303917,0.007671,3.375000,3.757991,10.795833,9.381279
3,1-1 place,31.291139,62.564490,1.348101,2.252245,2.383544,2.270612,0.740759,0.013143,3.493671,...,2.759167,3.366210,3.157500,3.374886,0.278833,0.006073,3.341667,3.735160,10.518333,9.402283
4,1-10 avenue,31.291139,62.564490,1.348101,2.252245,2.383544,2.270612,0.740759,0.013143,3.493671,...,2.759167,3.366210,3.157500,3.374886,0.278833,0.006073,3.341667,3.735160,10.518333,9.402283


In [17]:
# --- Melt from wide to long ---
street_weather_melted = street_weather.melt(
    id_vars=["normalized_street_name"],
    var_name="variable_season_year",
    value_name="value"
)

# --- Split variable, season, and year ---
street_weather_melted[["variable", "season", "year"]] = (
    street_weather_melted["variable_season_year"]
    .str.extract(r"^(\S+)\s+(\S+)\s+(\d{4})$")
)

# --- Rename roadwork → summer ---
street_weather_melted["season"] = street_weather_melted["season"].replace({"roadwork": "summer"})

# --- Construct combined key ---
street_weather_melted["normalized_street_name_season"] = (
    street_weather_melted["normalized_street_name"]
    + "_"
    + street_weather_melted["season"]
    + "_"
    + street_weather_melted["year"]
)

# --- Pivot back to wide, one row per street-season-year ---
street_weather_long = (
    street_weather_melted.pivot_table(
        index="normalized_street_name_season",
        columns="variable",
        values="value",
        aggfunc="first"
    )
    .reset_index()
)

# --- Clean column names ---
street_weather_long.columns.name = None


In [18]:
street_weather_long.head(20)

,normalized_street_name_season,precipitation,precipitation_hours,rain,snowfall,temperature,wind_speed
0,1-1 avenue loop_summer_2001,2.270612,2.465306,2.252245,0.013143,62.564490,10.024490
1,1-1 avenue loop_summer_2002,2.985714,2.828571,2.982857,0.002000,63.323265,10.443673
2,1-1 avenue loop_summer_2003,4.021633,4.195918,3.971837,0.035714,61.166122,9.971429
3,1-1 avenue loop_summer_2004,3.644898,3.391837,3.637551,0.005143,62.098776,9.839184
4,1-1 avenue loop_summer_2005,3.333469,3.228571,3.280816,0.038000,63.640816,10.209388
5,1-1 avenue loop_summer_2006,3.814694,3.481633,3.814694,0.000000,62.921633,10.349796
6,1-1 avenue loop_summer_2007,2.765306,2.759184,2.765306,0.000000,63.426939,9.968571
7,1-1 avenue loop_summer_2008,2.986939,2.722449,2.986531,0.000857,62.621224,9.947347
8,1-1 avenue loop_summer_2009,3.798367,4.179592,3.798367,0.000000,61.677959,10.000000
9,1-1 avenue loop_summer_2010,2.754694,2.918367,2.753878,0.000571,65.308980,10.742857


In [19]:
street_weather_long["snowfall"] = street_weather_long["snowfall"] * 10   # cm to mm

In [20]:
street_weather_long.to_csv("data/derived/street_weather_daily.csv", index = False)